<a href="https://colab.research.google.com/github/arnmon/Yolo_Flood_Depth/blob/main/IMDLIB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Connect Google drive with Google colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
pip install imdlib

In [3]:
# Download the IMD raw data into the google drive
import imdlib as imd
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon

path = "/content/drive/MyDrive/Colab Notebooks"

start_yr = 1900
end_yr = 2025
variable = 'rain' # other options are ('tmin'/ 'tmax')

imd.get_data(variable, start_yr, end_yr, fn_format='yearwise', file_dir=path)
data = imd.open_data(variable, start_yr, end_yr,'yearwise', path)
ds = data.get_xarray()
print(ds)



Downloading: rain for year 1900
Downloading: rain for year 1901
Downloading: rain for year 1902
Downloading: rain for year 1903
Downloading: rain for year 1904
Downloading: rain for year 1905
Downloading: rain for year 1906
Downloading: rain for year 1907
Downloading: rain for year 1908
Downloading: rain for year 1909
Downloading: rain for year 1910
Downloading: rain for year 1911
Downloading: rain for year 1912
Downloading: rain for year 1913
Downloading: rain for year 1914
Downloading: rain for year 1915
Downloading: rain for year 1916
Downloading: rain for year 1917
Downloading: rain for year 1918
Downloading: rain for year 1919
Downloading: rain for year 1920
Downloading: rain for year 1921
Downloading: rain for year 1922
Downloading: rain for year 1923
Downloading: rain for year 1924
Downloading: rain for year 1925
Downloading: rain for year 1926
Downloading: rain for year 1927
Downloading: rain for year 1928
Downloading: rain for year 1929
Downloading: rain for year 1930
Download

Exception: Error in file reading,mismatch in size of data-length

In [1]:
import imdlib as imd

path = "/content/drive/MyDrive/Colab Notebooks"
data = imd.open_data('rain', 2001, 2025, 'yearwise', path)
ds = data.get_xarray()
print(ds)

<xarray.Dataset> Size: 1GB
Dimensions:  (time: 9131, lat: 129, lon: 135)
Coordinates:
  * time     (time) datetime64[ns] 73kB 2001-01-01 2001-01-02 ... 2025-12-31
  * lat      (lat) float64 1kB 6.5 6.75 7.0 7.25 7.5 ... 37.75 38.0 38.25 38.5
  * lon      (lon) float64 1kB 66.5 66.75 67.0 67.25 ... 99.25 99.5 99.75 100.0
Data variables:
    rain     (time, lat, lon) float64 1GB -999.0 -999.0 -999.0 ... -999.0 -999.0
Attributes:
    Conventions:  CF-1.7
    title:        IMD gridded data
    source:       https://imdpune.gov.in/
    history:      2026-06-13 12:40:54.879191 Python
    references:   
    comment:      
    crs:          epsg:4326


In [2]:
# Provide the alttitude & Longitude of a point for which the data is required
#  And save the data into CSV file

lat = 28.46 #lattitude of point
lon = 77.05 #longitude of point
data.to_csv('data.csv', lat, lon, path)

In [ ]:
# Save CSV files for multiple points

# Provide lat and long in a list
latLong = [[20.3,77.23],[23.5,72.5],[26.0,77,1]]

for points in latLong:
  lat = points[0]
  lon = points[1]

  data.to_csv('test.csv', lat, lon, path)
  print ("data save for ",points)

In [ ]:
#  Provide the Geojson file of a catchment or polygon to dowlnaod all the gridded data lying into that polygon

geojson_file = '/content/drive/MyDrive/IMD/Test_geojson.geojson'


url="https://drive.google.com/file/d/111XvmUzvTlhY2lbFMseGVhZQh4pisFXQ/view?usp=sharing"
url2='https://drive.google.com/uc?id=' + url.split('/')[-2]

points_df = pd.read_csv(url2)


geometry = [Point(xy) for xy in zip(points_df['Long'], points_df['Lat'])]

# Creating the GeoDataFrame
gdf_points = gpd.GeoDataFrame(points_df, geometry=geometry)

# Set a CRS (coordinate reference system), EPSG:4326 is WGS84 Lat/Long
gdf_points.set_crs(epsg=4326, inplace=True)


gdf_polygon = gpd.read_file(geojson_file)

# Ensure both GeoDataFrames use the same CRS
if gdf_points.crs != gdf_polygon.crs:
    gdf_points = gdf_points.to_crs(gdf_polygon.crs)

gdf_list = []
for row in range (len(gdf_polygon)):
    points_in_polygon = gdf_points[gdf_points.within(gdf_polygon.iloc[row].geometry)]
    gdf_list.append(points_in_polygon)

final_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))

final_df = final_gdf[["Name","Lat","Long"]]
final_df.to_csv("Master_file.csv")

for index, row in final_df.iterrows():
    lat = row['Lat']
    lon = row['Long']
    data.to_csv('test.csv', lat, lon, path)
    print ("data save for " + str(lat)+ "_" +str(lon))